# Data Understanding
Profile every processed dataset before using it. Report missingness, duplicates, unique identifiers, category distributions, and text/numeric summaries; empty values in official metadata are expected where the source does not state a fact.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'data_quality_report.md'

In [ ]:
profiles = []
details = []
for path in sorted(DATA_DIR.glob('*.csv')):
    frame = pd.read_csv(path, keep_default_na=False)
    missing_frame = frame.replace('', pd.NA)
    id_column = frame.columns[0] if len(frame.columns) else None
    categorical_columns = [column for column in frame.select_dtypes(include=['object', 'category']).columns if frame[column].nunique() <= 40]
    profiles.append({
        'dataset': path.name,
        'rows_before_cleaning': len(frame),
        'rows_after_cleaning': len(frame),
        'columns': len(frame.columns),
        'duplicate_rows': int(frame.duplicated().sum()),
        'missing_values': int(missing_frame.isna().sum().sum()),
        'missing_percentage': round(float(missing_frame.isna().mean().mean() * 100), 2) if len(frame.columns) else 0.0,
        'unique_ids': int(frame[id_column].nunique()) if id_column else 0,
        'categorical_columns': ', '.join(categorical_columns),
    })
    details.append((path.name, frame))
summary = pd.DataFrame(profiles)
summary

In [ ]:
for name, frame in details:
    print(f'\n## {name}')
    display(frame.dtypes.to_frame('dtype'))
    missing_frame = frame.replace('', pd.NA)
    missing = missing_frame.isna().sum().to_frame('missing_values')
    missing['missing_percentage'] = (missing_frame.isna().mean() * 100).round(2)
    display(missing)
    print(f'Duplicate rows: {int(frame.duplicated().sum())}')
    categorical_columns = frame.select_dtypes(include=['object', 'category']).columns
    for column in categorical_columns:
        if frame[column].nunique(dropna=False) <= 40:
            print(f'Value counts: {column}')
            display(frame[column].value_counts(dropna=False).to_frame('count'))
    for column in frame.select_dtypes(include='number').columns:
        print(f'Numeric summary: {column}')
        display(frame[column].agg(['min', 'max', 'mean', 'median']).to_frame())
    for column in categorical_columns:
        if column.endswith('_id'):
            continue
        text = frame[column].fillna('').astype(str)
        lengths = text.str.len()
        whitespace_only = text.ne('') & text.str.strip().eq('')
        print(f'Text {column}: min={lengths.min() if len(lengths) else 0}, max={lengths.max() if len(lengths) else 0}, mean={lengths.mean() if len(lengths) else 0:.1f}, empty_strings={(text == "").sum()}, whitespace_only={whitespace_only.sum()}')

In [ ]:
from urllib.parse import urlsplit

URL_COLUMNS = {
    'official_sources.csv': ['url'],
    'service_catalog.csv': ['official_application_url', 'official_form_url', 'status_tracking_url', 'complaint_url', 'source_url'],
    'government_schemes.csv': ['application_url', 'source_url'],
    'documents_and_forms.csv': ['download_url', 'source_url'],
}
VALID_CATEGORIES = {
    'language': {'English', 'Telugu', 'Telugu-English', 'Unknown'},
    'intent': {'SERVICE_DISCOVERY', 'APPLICATION_PROCEDURE', 'DOCUMENT_REQUIREMENTS', 'ELIGIBILITY', 'FORM_DOWNLOAD', 'OFFICIAL_LINK', 'COMPLAINT_REGISTRATION', 'COMPLAINT_TRACKING', 'PAYMENT', 'SCHEME_DISCOVERY', 'SCHEME_ELIGIBILITY', 'STATUS_CHECK'},
    'complaint_category': {'Street Light', 'Road Damage', 'Garbage', 'Drainage', 'Water Supply', 'Water Leakage', 'Sewage', 'Illegal Dumping', 'Public Toilet', 'Traffic Signal', 'Stray Animals', 'Noise Complaint', 'Other Civic Issue'},
    'verification_status': {'unverified', 'verified', 'needs_review', 'unavailable'},
}
invalid_urls = 0
invalid_categories = 0
for filename, frame in details:
    for column in URL_COLUMNS.get(filename, []):
        if column in frame:
            for value in frame[column].astype(str):
                if value.strip():
                    parsed = urlsplit(value.strip())
                    invalid_urls += not (parsed.scheme in {'http', 'https'} and parsed.hostname)
    for column, allowed in VALID_CATEGORIES.items():
        if column in frame:
            invalid_categories += int((~frame[column].isin(allowed)).sum())

distribution_tables = []
for filename, column in [('citizen_queries.csv', 'language'), ('citizen_queries.csv', 'intent'), ('complaint_queries.csv', 'complaint_category')]:
    frame = dict(details)[filename]
    counts = frame[column].value_counts(dropna=False).rename_axis(column).reset_index(name='count')
    table_lines = ['| ' + ' | '.join(counts.columns) + ' |', '| --- | --- |']
    table_lines.extend('| ' + ' | '.join(str(value) for value in row) + ' |' for row in counts.itertuples(index=False, name=None))
    distribution_tables.extend([f'### {filename}: {column}', '', *table_lines, ''])

summary_table = ['| ' + ' | '.join(summary.columns) + ' |', '| ' + ' | '.join(['---'] * len(summary.columns)) + ' |']
summary_table.extend('| ' + ' | '.join(str(value) for value in row) + ' |' for row in summary.itertuples(index=False, name=None))
report_lines = [
    '# Data Quality Report', '',
    'Generated from current processed CSVs. Rows before/after cleaning match because these files are already the cleaned outputs.', '',
    *summary_table, '',
    f'- Invalid URLs: {invalid_urls}',
    f'- Invalid categories/language/intent labels: {invalid_categories}', '',
    '## Language, Intent, and Complaint Category Distributions', '',
    *distribution_tables,
    '## Data Quality Issues',
    '- Official metadata is limited to names and listing links; unsupported detail fields remain blank.',
    '- Direct form and scheme destinations need individual review.',
    '- Synthetic examples do not represent measured citizen behavior.', '',
    '## Cleaning Actions',
    '- Synthetic datasets are normalized, deduplicated, validated, and privacy-scanned before saving.',
    '- Telugu Unicode and code-mixed text are preserved.', '',
    '## Remaining Limitations',
    '- Source pages may change; each service detail and destination link requires its own review.',
    '- TGDeX record-level health/demographic datasets are excluded pending privacy review.',
]
REPORT_PATH.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print(f'Wrote {REPORT_PATH}; invalid_urls={invalid_urls}; invalid_categories={invalid_categories}')